In [1]:
import torch
import torchvision
import torchvision.transforms as transforms

/home/otto/attention/src/attention/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
if torch.cuda.is_available():
    device = 'cuda:0'
else:
    device = 'cpu'

In [3]:
batch_size = 4

transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

Files already downloaded and verified
Files already downloaded and verified


$$ 
q(W_l^Q) = \mathcal{N}(\mu_{_l}^Q, \Sigma_{l}^Q) \quad \Sigma_{l}^Q = (U_lU_l^T) \otimes (V_lV_l^T) + \sum_h S_{l,h} \otimes I + \text{diag}(D_l)
$$

In [4]:
from vit import ViT
from metrics import *

vit = ViT(image_size=32, patch_size=4, num_classes=len(classes), dim=128, depth=6, heads=8, mlp_dim=128*4).to(device)

In [5]:
from torch.optim import Adam
import torch.nn.functional as F

lr = 3e-4
optim = Adam(vit.parameters(), lr=3e-4)

nepochs = 10
from tqdm import tqdm

for epoch in range(nepochs):
    loss_epochs = 0
    vit.train()
    for images, labels in tqdm(trainloader):
        images, labels = images.to(device), labels.to(device)
        logits = vit(images)
        loss = F.cross_entropy(logits, labels)

        loss.backward()
        optim.step()
        optim.zero_grad()

        loss_epochs += loss.item()
        
    rob = robustness_vs_noise(vit, testloader, device, bayes=False)
    print(loss_epochs / len(trainloader))
    print(rob)
        


100%|██████████| 2500/2500 [00:29<00:00, 85.94it/s]


1.5351453230643273
{0.0: 0.496, 0.1: 0.4621, 0.25: 0.3642, 0.4: 0.2925}


100%|██████████| 2500/2500 [00:29<00:00, 84.71it/s]


1.2822406332558394
{0.0: 0.5313, 0.1: 0.5093, 0.25: 0.4069, 0.4: 0.298}


100%|██████████| 2500/2500 [00:29<00:00, 85.98it/s]


...

0.7403021557809413
{0.0: 0.6245, 0.1: 0.5247, 0.25: 0.3186, 0.4: 0.2027}


100%|██████████| 2500/2500 [00:29<00:00, 86.18it/s]


0.6689102001828886
{0.0: 0.6321, 0.1: 0.5432, 0.25: 0.3261, 0.4: 0.2154}


100%|██████████| 2500/2500 [00:29<00:00, 84.98it/s]

0.6029493777824193
{0.0: 0.6426, 0.1: 0.5688, 0.25: 0.3578, 0.4: 0.2375}


In [2]:
import bayesian_vit as bayes

bvit = bayes.BViT(image_size=32, patch_size=4, num_classes=10, dim=128, depth=6, heads=8, mlp_dim=128*4,
           qkv_rank=3, prior_var=1.0).to(device)

NameError: name 'Attention' is not defined

In [ ]:
def train_epoch(vit_model, trainloader, optimizer, device, dataset_size, kl_scale=1.0):
    vit_model.train()
    total_loss = 0.0
    for images, labels in tqdm(trainloader):
        images = images.to(device)
        labels = labels.to(device)

        logits = vit_model(images, sample=True)  # sample weights each forward
        nll = F.cross_entropy(logits, labels, reduction='mean')  # batch mean NLL

        # analytical KL from all BayesianLinear layers
        kl = bayes.model_kl_loss(vit_model)

        # scale KL by 1/N: standard VI ELBO scaling when using mini-batches
        loss = nll + (kl_scale * kl) / float(dataset_size)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(trainloader)



In [ ]:
dataset_size = len(trainset)
optim = Adam(bvit.parameters(), lr=3e-4)

for epoch in range(nepochs):
    loss_epoch = train_epoch(bvit, trainloader, optim, device, dataset_size, kl_scale=1.0)
    #metrics = evaluate(bvit, testloader, device)
    rob = robustness_vs_noise(bvit, testloader, device)
    print(epoch, loss_epoch, rob)